# 01 — Data Loading, Cleaning & Merging

**Purpose**: Load IMDb TSV files and TMDb Kaggle dataset, clean, merge, and save unified dataframe.

**Outputs**:
- `data/movies_clean.parquet` — merged movie dataframe
- `data/actor_popularity.parquet` — actor popularity scores
- `data/director_stats.parquet` — director performance stats
- `data/valid_tconst_set.parquet` — valid movie IDs

**Key decisions** (see wiki Decision Log):
- D007: TMDb data from Kaggle alanvourch dataset (not API)
- D010: Min vote threshold configurable, default 500
- D012: UV as package manager
- D014: isAdult filter (stakeholder discussion point)

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import warnings
import re
warnings.filterwarnings('ignore')

# Paths
SOURCE_DIR = os.path.join('..', 'source_data')
DATA_DIR = os.path.join('..', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

# Config (D010: configurable min vote threshold)
MIN_VOTE_THRESHOLD = 500
CURRENT_YEAR = 2025

print('Config:')
print(f'  MIN_VOTE_THRESHOLD = {MIN_VOTE_THRESHOLD}')
print(f'  CURRENT_YEAR = {CURRENT_YEAR}')
print(f'  SOURCE_DIR = {os.path.abspath(SOURCE_DIR)}')
print(f'  DATA_DIR = {os.path.abspath(DATA_DIR)}')

## 1. Load IMDb Datasets

Loading 5 TSV files from IMDb bulk data. Key notes:
- `\N` = null in IMDb format → replace with NaN
- Year columns (startYear, endYear) are float due to NaN → convert to nullable Int64
- Filter `titleType == 'movie'` (exclude shorts, TV, etc.)
- Filter `isAdult` to valid values only (0 or 1)

In [ ]:
def load_imdb_tsv(filename, source_dir=SOURCE_DIR, usecols=None):
    """Load an IMDb TSV.gz file, replace \\N with NaN, fix year columns."""
    path = os.path.join(source_dir, filename)
    print(f'Loading {filename}...')
    df = pd.read_csv(path, sep='\t', compression='gzip', 
                     na_values='\\N', low_memory=False, usecols=usecols)
    
    # Fix year columns (float → nullable Int64)
    for col in ['startYear', 'endYear', 'birthYear', 'deathYear']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Nulls:\n{df.isnull().sum()}')
    display(df.head(3))
    return df

In [ ]:
# Load title.basics — movie metadata
title_basics = load_imdb_tsv('title.basics.tsv.gz')

# Print all titleType values (for inspection)
print('\n=== titleType values ===')
print(title_basics['titleType'].value_counts().to_string())

# Fix isAdult: filter to valid values only (0 or 1)
# NOTE: isAdult has corrupt data — year values (2023, 1979) mixed with booleans (0, 1)
print('\n=== isAdult values (before fix) ===')
print(title_basics['isAdult'].value_counts().head(20).to_string())

valid_adult = title_basics['isAdult'].isin([0, 1])
print(f'\nRows with valid isAdult: {valid_adult.sum():,} / {len(title_basics):,}')
title_basics = title_basics[valid_adult].copy()

# D014: Exclude adult content (3.3% of dataset)
# Stakeholder discussion: "We excluded adult content as it's not relevant for mainstream movie studio recommendations"
print('\n=== isAdult values (after fix) ===')
print(title_basics['isAdult'].value_counts().to_string())

In [ ]:
# Filter to movies only
print(f'Before movie filter: {len(title_basics):,} rows')
title_basics = title_basics[title_basics['titleType'] == 'movie'].copy()
print(f'After movie filter: {len(title_basics):,} rows')
print(f'Start year range: {title_basics["startYear"].min()} - {title_basics["startYear"].max()}')
display(title_basics.head(3))

In [ ]:
# Load title.ratings — ratings and vote counts
title_ratings = load_imdb_tsv('title.ratings.tsv.gz')

# Filter by vote threshold (D010)
print(f'Before vote filter: {len(title_ratings):,} rows')
title_ratings = title_ratings[title_ratings['numVotes'] >= MIN_VOTE_THRESHOLD].copy()
print(f'After vote filter (>= {MIN_VOTE_THRESHOLD}): {len(title_ratings):,} rows')

In [ ]:
# Compute valid tconst set = intersection of filtered title_basics and title_ratings
valid_tconsts = set(title_basics['tconst']) & set(title_ratings['tconst'])
print(f'Valid tconst set: {len(valid_tconsts):,} movies (all years, >= {MIN_VOTE_THRESHOLD} votes)')

# Save valid_tconst_set for EDA notebook
valid_tconst_df = pd.DataFrame({'tconst': list(valid_tconsts)})
valid_tconst_df.to_parquet(os.path.join(DATA_DIR, 'valid_tconst_set.parquet'), index=False)
print('Saved valid_tconst_set.parquet')

In [ ]:
# Load title.principals in chunks (788MB file — too large to load entirely)
# Filter to valid tconsts AND actor/actress category
print('Loading title.principals.tsv.gz (chunked)...')
chunks = []
for i, chunk in enumerate(pd.read_csv(
    os.path.join(SOURCE_DIR, 'title.principals.tsv.gz'),
    sep='\t', compression='gzip', na_values='\\N',
    chunksize=1_000_000
)):
    chunk = chunk[
        (chunk['tconst'].isin(valid_tconsts)) & 
        (chunk['category'].isin(['actor', 'actress']))
    ]
    chunks.append(chunk)
    if (i + 1) % 10 == 0:
        print(f'  Processed {(i+1) * 1_000_000:,} rows...')

title_principals = pd.concat(chunks, ignore_index=True)
del chunks  # Free memory

print(f'\nFinal shape: {title_principals.shape}')
print(f'Unique actors: {title_principals["nconst"].nunique():,}')
print(f'Unique movies: {title_principals["tconst"].nunique():,}')
display(title_principals.head(3))

In [ ]:
# Load title.crew — director and writer IDs
# Filter to valid tconst set only
title_crew = load_imdb_tsv('title.crew.tsv.gz')
title_crew = title_crew[title_crew['tconst'].isin(valid_tconsts)].copy()
print(f'After filter to valid tconsts: {len(title_crew):,} rows')

In [ ]:
# Load name.basics — person metadata
# Filter to only actors/actresses in our principals dataset
name_basics = load_imdb_tsv('name.basics.tsv.gz')

actor_nconsts = set(title_principals['nconst'])
print(f'Actor nconsts from principals: {len(actor_nconsts):,}')

name_basics = name_basics[name_basics['nconst'].isin(actor_nconsts)].copy()
print(f'After filter to relevant actors: {len(name_basics):,} rows')
display(name_basics.head(3))

## 2. Load TMDb Kaggle Dataset (D007)

alanvourch's "Ultimate 1M Movies Dataset (TMDB + IMDb)" from Kaggle.
- Has `imdb_id` for joining with IMDb data
- Provides budget, revenue, cast, director, genres
- Filtered to valid tconst set to reduce memory

In [ ]:
# Load TMDb CSV from zip
tmdb_zip = os.path.join(SOURCE_DIR, 'alanvourch-tmdb-movies-daily-updates.zip')
print(f'Loading TMDb from {tmdb_zip}...')

with zipfile.ZipFile(tmdb_zip) as z:
    with z.open('TMDB_all_movies.csv') as f:
        tmdb = pd.read_csv(f, low_memory=False)

# Rename and filter
tmdb = tmdb.rename(columns={'imdb_id': 'tconst'})
tmdb_cols = ['tconst', 'budget', 'revenue', 'popularity', 'production_companies', 
             'production_countries', 'certification_us', 'keywords']
tmdb = tmdb[tmdb_cols].drop_duplicates(subset='tconst', keep='first')
tmdb = tmdb[tmdb['tconst'].isin(valid_tconsts)].copy()
print(f'TMDb after filter: {len(tmdb):,} rows')

# Free memory
del valid_tconst_df
print('Freed: valid_tconst_df')

## 3. Merge Datasets

Merge order:
1. `title_basics` + `title_ratings` → `movies`
2. `movies` + `title_crew` → add director/writer IDs
3. `movies` + `name_basics` → add director names
4. `movies` + `tmdb` → add budget, revenue, etc.

In [ ]:
# Step 1: Merge basics + ratings
movies = title_basics.merge(title_ratings, on='tconst', how='inner')
print(f'After basics + ratings merge: {len(movies):,} rows')

# Step 2: Merge with crew (directors)
movies = movies.merge(title_crew, on='tconst', how='left')
movies['primary_director_nconst'] = movies['directors'].str.split(',').str[0]

# Step 3: Merge with name_basics to get director name
director_names = name_basics[['nconst', 'primaryName']].rename(
    columns={'nconst': 'primary_director_nconst', 'primaryName': 'director_name'}
)
movies = movies.merge(director_names, on='primary_director_nconst', how='left')
print(f'After crew merge: {len(movies):,} rows')
print(f'Has director name: {movies["director_name"].notna().sum():,}')

# Step 4: Merge with TMDb
movies = movies.merge(tmdb, on='tconst', how='left')
print(f'After TMDb merge: {len(movies):,} rows')
print(f'Has budget > 0: {(movies["budget"] > 0).sum():,}')
print(f'Has revenue > 0: {(movies["revenue"] > 0).sum():,}')

# Free memory: title_basics, title_crew, tmdb, director_names
del title_basics, title_crew, tmdb, director_names
print('Freed: title_basics, title_crew, tmdb, director_names')

display(movies[['tconst', 'primaryTitle', 'startYear', 'averageRating', 'numVotes', 'director_name', 'budget', 'revenue']].head(3))

## 4. Compute Actor Popularity (D008, D009)

Formula: `Actor Popularity = SUM( (1/log(ordering+1)) × numVotes × averageRating )`

Computed across ALL movies (historical data) — not just 2023.

In [ ]:
# Merge principals with ratings
principals_with_ratings = title_principals.merge(title_ratings, on='tconst', how='left')

# Compute billing weight (D009: 1/log(ordering+1))
principals_with_ratings['billing_weight'] = 1.0 / np.log(
    principals_with_ratings['ordering'].astype(float) + 1
)

# Compute weighted score per appearance
principals_with_ratings['weighted_score'] = (
    principals_with_ratings['billing_weight'] * 
    principals_with_ratings['numVotes'].fillna(0) * 
    principals_with_ratings['averageRating'].fillna(0)
)

# Aggregate by actor
actor_popularity = principals_with_ratings.groupby('nconst').agg(
    total_score=('weighted_score', 'sum'),
    num_movies=('tconst', 'nunique'),
    total_votes=('numVotes', 'sum')
).reset_index()

# Add actor names
actor_names = name_basics[['nconst', 'primaryName']].drop_duplicates()
actor_popularity = actor_popularity.merge(actor_names, on='nconst', how='left')

# Sort by popularity
actor_popularity = actor_popularity.sort_values('total_score', ascending=False)

print(f'Top 20 Most Popular Actors (across all years in dataset):')
display(actor_popularity.head(20)[['primaryName', 'total_score', 'num_movies', 'total_votes']])

# Free memory: title_principals, title_ratings, name_basics, principals_with_ratings, actor_names
del title_principals, title_ratings, name_basics, principals_with_ratings, actor_names
print('Freed: title_principals, title_ratings, name_basics, principals_with_ratings, actor_names')

## 5. Compute Director Performance

In [ ]:
# Director stats from filtered movies
director_stats = movies.groupby('director_name').agg(
    num_movies=('tconst', 'nunique'),
    avg_rating=('averageRating', 'mean'),
    avg_votes=('numVotes', 'mean'),
    total_votes=('numVotes', 'sum')
).reset_index()

director_stats = director_stats.sort_values('total_votes', ascending=False)

print(f'Top 20 Directors by Total Audience Reach:')
display(director_stats.head(20))

## 6. Save All Dataframes

In [ ]:
# Save all dataframes
movies.to_parquet(os.path.join(DATA_DIR, 'movies_clean.parquet'), index=False)
actor_popularity.to_parquet(os.path.join(DATA_DIR, 'actor_popularity.parquet'), index=False)
director_stats.to_parquet(os.path.join(DATA_DIR, 'director_stats.parquet'), index=False)

print('Saved:')
print(f'  movies_clean.parquet ({os.path.getsize(os.path.join(DATA_DIR, "movies_clean.parquet")) / 1024 / 1024:.1f} MB)')
print(f'  actor_popularity.parquet ({os.path.getsize(os.path.join(DATA_DIR, "actor_popularity.parquet")) / 1024 / 1024:.1f} MB)')
print(f'  director_stats.parquet ({os.path.getsize(os.path.join(DATA_DIR, "director_stats.parquet")) / 1024 / 1024:.1f} MB)')

## 7. Quick 2023 Snapshot

Preview 2023 movies for EDA Q1 and Q2.

In [ ]:
# 2023 movies (filter from already-loaded data)
movies_2023 = movies[movies['startYear'] == 2023].copy()
print(f'2023 movies with >= {MIN_VOTE_THRESHOLD} votes: {len(movies_2023):,}')

# Top 10 by rating
print(f'\nTop 10 Highest Rated 2023 Movies:')
display(movies_2023.nlargest(10, 'averageRating')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

# Top 10 by votes (most popular)
print(f'\nTop 10 Most Voted 2023 Movies:')
display(movies_2023.nlargest(10, 'numVotes')[
    ['primaryTitle', 'averageRating', 'numVotes', 'director_name']
])

In [ ]:
# Final summary
print('='*60)
print('FINAL DATAFRAME SUMMARY')
print('='*60)
print(f'Shape: {movies.shape}')
print(f'\nColumns: {list(movies.columns)}')
print(f'\nDtypes:\n{movies.dtypes}')
print(f'\nNull counts:\n{movies.isnull().sum()}')
print(f'\nSample (top rated):')
display(movies.nlargest(5, 'averageRating')[
    ['tconst', 'primaryTitle', 'startYear', 'averageRating', 'numVotes', 'director_name', 'budget', 'revenue']
])